# Solving the 1D heat equation: PINN, spline PIKAN, and finite differences

This notebook implements the comparison proposed in `PIKAN_Heat_Equation_Academic_Project_Proposal.pdf` and adds a classical Crank–Nicolson finite difference solver. Both networks learn the same initial–boundary value problem using physics losses and **no interior temperature labels**. The exact solution is reserved for validation and the final evaluation.

**Study scope:** This is a controlled implementation exercise and architecture comparison for one smooth PDE instance, not a practical speed benchmark for general heat solvers. Because this particular case has a closed-form solution, direct formula evaluation is also benchmarked below.

Run top to bottom in the `ml` environment. The default experiment uses CPU so the timing comparison uses the same hardware for every run. For a faster draft, reduce `STEPS` and/or `SEEDS` below; rerun all cells before interpreting the conclusions.

**Problem:** $T_t=\alpha T_{xx}$ on $(x,t)\in[0,1]^2$, $\alpha=0.1$, $T(x,0)=\sin(\pi x)$, $T(0,t)=T(1,t)=0$. The exact solution is $T(x,t)=e^{-\alpha\pi^2t}\sin(\pi x)$.

In [1]:
import math
import time
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn

torch.set_num_threads(min(4, torch.get_num_threads()))
DEVICE = torch.device("cpu")
DTYPE = torch.float32
ALPHA = 0.1
STEPS = 400
SEEDS = (0, 1, 2, 3, 4)
N_INTERIOR, N_INITIAL, N_BOUNDARY = 96, 48, 48
LR = 3e-3
WEIGHTS = {"physics": 1.0, "initial": 10.0, "boundary": 10.0}
print(f"PyTorch {torch.__version__}; device={DEVICE}; steps={STEPS}; seeds={SEEDS}")

PyTorch 2.13.0+cu130; device=cpu; steps=400; seeds=(0, 1, 2, 3, 4)


## 1. Exact solution and collocation points

Each seed generates one fixed collection of uniformly sampled interior, initial, and boundary points. The MLP and KAN for that seed receive **identical points**, weights, Adam settings, and update counts. Fixed points make this small comparison easy to reproduce; they also mean the result measures fit to one collocation design per seed.

In [2]:
def exact_temperature(coords):
    x, t = coords[:, :1], coords[:, 1:2]
    return torch.exp(-ALPHA * math.pi**2 * t) * torch.sin(math.pi * x)


def sample_points(seed):
    rng = torch.Generator(device="cpu").manual_seed(seed)
    interior = torch.rand(N_INTERIOR, 2, generator=rng, dtype=DTYPE)
    initial = torch.cat((torch.rand(N_INITIAL, 1, generator=rng, dtype=DTYPE),
                         torch.zeros(N_INITIAL, 1, dtype=DTYPE)), dim=1)
    boundary = torch.rand(N_BOUNDARY, 2, generator=rng, dtype=DTYPE)
    boundary[:, 0] = torch.arange(N_BOUNDARY) % 2
    return interior.to(DEVICE), initial.to(DEVICE), boundary.to(DEVICE)


def pde_residual(model, coords, create_graph=True):
    coords = coords.detach().clone().requires_grad_(True)
    value = model(coords)
    first = torch.autograd.grad(value, coords, torch.ones_like(value), create_graph=True)[0]
    second_x = torch.autograd.grad(first[:, :1], coords,
                                   torch.ones_like(first[:, :1]),
                                   create_graph=create_graph)[0][:, :1]
    return first[:, 1:2] - ALPHA * second_x


class ExactModel(nn.Module):
    def forward(self, coords):
        return exact_temperature(coords)


probe = torch.rand(64, 2, generator=torch.Generator().manual_seed(2026))
exact_residual = pde_residual(ExactModel(), probe, create_graph=False)
assert exact_residual.abs().max() < 2e-6
print("Exact-solution PDE residual, max absolute:", f"{exact_residual.abs().max().item():.2e}")

Exact-solution PDE residual, max absolute: 1.19e-07


## 2. Architectures

The PINN is a two hidden layer tanh MLP. Each PIKAN edge has a trainable scalar base branch plus a trainable **cubic B-spline** expansion; node outputs sum their incoming edge functions. The spline knots are fixed and shared across edges, and hidden nodes use tanh to keep their inputs in the spline range. `grid_size` is the number of spline centers per edge. This is a **KAN-style variant** with genuine learned univariate edge functions. The added hidden tanh and fixed grid mean its results should not be generalized to every KAN implementation.

The MLP width is chosen to nearly match the trainable parameter count of the 8-center PIKAN. The 12-center KAN is an additional grid-size sensitivity check and therefore has more parameters.

In [3]:
def cubic_spline_basis(z):
    # Centered cardinal cubic B-spline, support [-2, 2], C2 continuous.
    a = z.abs()
    inner = 2.0 / 3.0 - a.square() + 0.5 * a.pow(3)
    outer = (2.0 - a).clamp_min(0).pow(3) / 6.0
    return torch.where(a < 1.0, inner, outer)


class SplineKANLayer(nn.Module):
    def __init__(self, in_features, out_features, grid_size=8):
        super().__init__()
        centers = torch.linspace(-1.25, 1.25, grid_size)
        self.register_buffer("centers", centers)
        self.spacing = float(centers[1] - centers[0])
        self.base = nn.Linear(in_features, out_features)
        self.spline = nn.Parameter(torch.empty(out_features, in_features, grid_size))
        nn.init.normal_(self.spline, std=0.02)

    def forward(self, inputs):
        z = (inputs.unsqueeze(-1) - self.centers) / self.spacing
        basis = cubic_spline_basis(z)
        return self.base(torch.nn.functional.silu(inputs)) + torch.einsum("nig,oig->no", basis, self.spline)


class PIKAN(nn.Module):
    def __init__(self, grid_size=8):
        super().__init__()
        self.layers = nn.ModuleList([
            SplineKANLayer(2, 6, grid_size),
            SplineKANLayer(6, 6, grid_size),
            SplineKANLayer(6, 1, grid_size),
        ])

    def forward(self, coords):
        values = 2 * coords - 1  # physical [0, 1] coordinates to spline input range
        for layer in self.layers[:-1]:
            values = torch.tanh(layer(values))
        return self.layers[-1](values)


class PINN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(2, 20), nn.Tanh(),
                                 nn.Linear(20, 20), nn.Tanh(), nn.Linear(20, 1))

    def forward(self, coords):
        return self.net(2 * coords - 1)


def parameter_count(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


for name, model in [("PINN", PINN()), ("PIKAN-8", PIKAN(8)), ("PIKAN-12", PIKAN(12))]:
    print(f"{name}: {parameter_count(model)} trainable parameters")

PINN: 501 trainable parameters
PIKAN-8: 499 trainable parameters
PIKAN-12: 715 trainable parameters


### Derivative check

The PDE loss depends on second spatial derivatives. Check the spline branch's autograd curvature against a finite difference at a point away from knot boundaries, then check that both full models permit second derivatives with respect to physical coordinates.

In [4]:
z = torch.tensor([0.37], dtype=torch.float64, requires_grad=True)
first = torch.autograd.grad(cubic_spline_basis(z), z, create_graph=True)[0]
second = torch.autograd.grad(first, z)[0].item()
h = 1e-3
finite_difference = ((cubic_spline_basis(z.detach() + h)
                      - 2 * cubic_spline_basis(z.detach())
                      + cubic_spline_basis(z.detach() - h)) / h**2).item()
assert abs(second - finite_difference) < 2e-3
for model in (PINN(), PIKAN(8)):
    residual = pde_residual(model, probe[:8], create_graph=False)
    assert torch.isfinite(residual).all()
print(f"Spline B''(0.37): autograd={second:.6f}, finite difference={finite_difference:.6f}; full-model derivatives finite")

Spline B''(0.37): autograd=-0.890000, finite difference=-0.890000; full-model derivatives finite
